# Chapter 25: AI Agents. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/25-robots.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/25-robots.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/25-robots.ipynb)

In [ ]:
# The exercises of this notebook need no package besides base R

## Exercise 25.9: Building a Simple Agent Loop

Implement a deterministic conversion controller in R using structured `(value, unit, target)` inputs.

Core. Submit a returned action log, terminal status, and three test results.

a) Define `calculator(expression)` for trusted expressions and `unit_convert(value, from, to)`. Use 1 cup = 236.6 mL, 1 L = 1000 mL, and 1 kg = 2.205 lbs.

b) Follow cups $\to$ mL $\to$ L or kg $\to$ lbs, updating state after each tool observation. Stop within five steps. Return an action log and status (`complete`, `unsupported`, or `step_limit`), with an answer only on completion. Cross-check completed conversions with the calculator.

c) Test `(2.5, "cups", "L")`, `(4, "kg", "lbs")`, and `(1, "m", "L")`. Report calls and terminal statuses.

d) What component is missing for LLM-driven ReAct? Optionally replace the controller with model-selected structured actions, retaining tool restrictions and a five-call model budget.

## Exercise 25.10: Agent Cost and Performance Analysis

Use hypothetical prices of \$0.003/1K input tokens and \$0.015/1K output tokens. The data frame `tasks` simulates 50 runs with call counts, tokens per call, and binary success indicators.

Core. Submit plots, cost metrics, and savings under stated assumptions.

In [ ]:
# Simulated agent runs on 50 tasks
set.seed(42)
n_tasks <- 50

# Each task has: n_llm_calls, input_tokens, output_tokens, n_tool_calls, success
tasks <- data.frame(
  task_id = 1:n_tasks,
  n_llm_calls = rpois(n_tasks, 4) + 1,
  avg_input_tokens = round(rnorm(n_tasks, 800, 200)),
  avg_output_tokens = round(rnorm(n_tasks, 200, 80)),
  n_tool_calls = rpois(n_tasks, 2)
)
tasks$avg_input_tokens <- pmax(tasks$avg_input_tokens, 100)
tasks$avg_output_tokens <- pmax(tasks$avg_output_tokens, 50)

tasks$success <- rbinom(n_tasks, 1, plogis(-3 + 0.8 * tasks$n_llm_calls))

a) Compute and plot task cost $C_i$: input + output + tools at \$0.01 per call.

b) With success indicator $Y_i$, distinguish mean successful-run cost $\sum_i C_iY_i/\sum_iY_i$ from expenditure per success $\sum_i C_i/\sum_iY_i$. Also compute mean failed-run cost. Which metric includes failed attempts?

c) Describe the associations between LLM-call count and success, and between cost and success.

d) Save one LLM call per task with tokens per call, tool calls, and outcomes fixed. Compute total savings. What measurement would support a claim that better prompting achieves them?

e) Inspect $P(Y_i=1\mid n_i)=\operatorname{logit}^{-1}(-3+0.8n_i)$, where $n_i$ counts calls. Which associations are assumed? How does reducing $n_i$ differ from holding outcomes fixed?

## Exercise 25.11: Multi-Agent Simulation

Simulate code review with a Writer producing a random quality score and a Reviewer providing feedback. Scores, improvements, and prices are modeling assumptions.

Core. Submit plots, budget comparisons, and a sensitivity table. Use `set.seed(42)`.

a) The Writer's initial quality is $q \sim N(0.6,0.15^2)$, clipped to $[0,1]$. An ideal Reviewer observes $q$ exactly and accepts if $q>0.8$. After a failed check, revise by adding $\Delta q\sim N(0.15,0.05^2)$, capped at 1.0. Simulate 100 tasks with at most five checks and four revisions. Mark a task failed if its fifth check fails.

b) Plot accepted tasks' check counts and failures separately. What fractions of all tasks are accepted within one check and within three?

c) At \$0.05 per generation and check, compare cost, outputs above 0.8, and final-quality distributions with one solo attempt per task, $q\sim N(0.7,0.15^2)$, clipped to $[0,1]$. The solo model's higher initial mean is assumed. Why does this comparison not isolate coordination's benefit?

d) Match budgets: give each solo task as many independent attempts as the multi-agent task used checks, at the same price per generation and check. Using the same ideal quality oracle, select the highest $q$. Compare true quality and cost.

e) Vary mean revision gain $\mu\in\{0,0.15\}$ and scoring noise $\sigma\in\{0,0.15\}$. Score each check as $q+\epsilon$, with independent $\epsilon\sim N(0,\sigma^2)$. Use noisy scores for review acceptance and solo selection, but assess outcomes by true $q>0.8$. Recompute matched budgets. Does the ranking reverse anywhere? Explain.

## Exercise 25.12: Red-Teaming an Agent: Screening Suspicious Text

The chapter distinguishes three vulnerability categories: *external prompt injection* (malicious instructions embedded in emails, ads, or web pages), *agent mistakes* (harmful actions arising from reasoning errors), and *direct misuse* (a user requesting a prohibited action). Consider an agent that can browse the web, edit spreadsheets, and send email. These are evaluated with a two-stage pipeline: an automated scanner flags potential breaches, then human experts review only the flagged cases, as in the chapter's Toloka case study (Example 25.4). This exercise screens scenario text for the first stage; it does not execute an agent or measure breaches.

Core. Submit a labeled corpus, held-out confusion matrices, and a review budget analysis using base R.

a) Give one scenario per category: environment, triggering content, authorization policy, and harmful action. Distinguish attack attempts from harmful outcomes.

b) Build 26 short descriptions with explicit policy and authorization context: for external injection and agent mistakes, five containing prohibited instructions or describing harmful proposed actions (positive text labels) and four benign look-alikes each; for direct misuse, four positive and four benign. Randomly reserve two positive and two benign examples per category (12 texts) before developing a single rule-based scanner on the other 14. Freeze its rules; report held-out confusion matrices, precision, recall, and F1 per category. Recall is $P(\text{flagged}\mid\text{positive text label})$. Mark zero-denominator ratios undefined.

c) Aggregate flagged and missed positives and flagged benign texts. If humans review only flagged texts, how many reviews are requested and which examples never reach them?

d) Relate missed positives to category counts and recall. How would changing category prevalence alter budget priorities at unchanged recall? Explain these tiny samples' limits, propose an audit of unflagged text, and specify traces and outcome labels needed to measure harmful actions and legitimate-task success.